In [1]:
import hanlp

# Pipeline allows blending multiple callable functions no matter they are a rule, a TensorFlow component or a PyTorch
# one. However, it's slower than the MTL framework.
# pos = hanlp.load(hanlp.pretrained.pos.CTB9_POS_ALBERT_BASE)  # In case both tf and torch are used, load tf first.

HanLP = hanlp.pipeline()\
    .append(hanlp.load('CTB9_TOK_ELECTRA_BASE_CRF'), output_key='tok') \
    .append(hanlp.load('CTB9_POS_ELECTRA_SMALL_TF'), output_key='pos') \
    .append(hanlp.load('CTB9_DEP_ELECTRA_SMALL', conll=0), output_key='dep', input_key='tok')

In [2]:
from YSUtils import *
from collections import defaultdict
import numpy as np
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.datasets import load_iris
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm

In [ ]:
output = []
for txt_file in tqdm(list_txt_files("./LCMC_all_txt/")):
    lines = []
    with open(txt_file, "r", encoding="utf-8") as file:
        for line in file:
            if line == '\n' or len(line.strip()) > 128:
                continue
            lines.append(line.strip())
    doc = HanLP(lines)
    
    
    for i, line in enumerate(lines):
        line_json = {}
        line_json["text"] = line.strip()
        line_deps_with_word = []
        for j, dep_couple in enumerate(doc["dep"][i]):
            str_with_word =  doc["pos"][i][j] + ' ' + doc["pos"][i][dep_couple[0] - 1] + ' ' + dep_couple[1] + ' ' + doc["tok"][i][j] + ' ' + doc["tok"][i][dep_couple[0] - 1] + ' ' + dep_couple[1]
            # str_dep = doc["pos/ctb"][i][j] + ' ' + doc["pos/ctb"][i][dep_couple[0] - 1] + ' ' + dep_couple[1] + ' ' + doc["tok/coarse"][i][j] + ' ' + doc["tok/coarse"][i][j] + ' ' + dep_couple[1]
            line_deps_with_word.append(str_with_word)
        line_json["deps_with_word"] = line_deps_with_word
        output.append(line_json)
with open("./dep_feature/LCMC_all_dep.json", "w", encoding="utf-8") as file:
    file.write(json.dumps(output, ensure_ascii=False))

    

In [5]:
feature_count = defaultdict(int)
with open('./dep_feature/LCMC_all_dep.json', 'r', encoding='utf-8') as file:
    origin_dep_feature = json.load(file)
for line in origin_dep_feature:
    for dep in line["deps_with_word"]:
        dep_list = dep.split()
        dep_feature = dep_list[0] + " " + dep_list[1] + " " + dep_list[2]
        feature_count[dep_feature] += 1

with open('./dep_feature/20250307_Yiyan_gpt_all_dep.json', 'r', encoding='utf-8') as file:
    instruct_dep_feature = json.load(file)
for line in instruct_dep_feature:
    for dep in line["deps_with_word"]:
        dep_list = dep.split()
        # 将前三个元素组成字符串
        dep_feature = dep_list[0] + " " + dep_list[1] + " " + dep_list[2]
        feature_count[dep_feature] += 1

sorted_feature_count = dict(sorted(feature_count.items(), key=lambda x: x[1], reverse=True))
with open('./dep_feature/dep_feature_count.json', 'w', encoding='utf-8') as file:
    file.write(json.dumps(sorted_feature_count, ensure_ascii=False))

In [10]:
with open('./dep_feature/dep_feature_count.json', 'r', encoding='utf-8') as file:
    dep_feature_count = json.load(file)
    # 筛选出大于***的键值对
    dep_feature_index = {k: v for k, v in dep_feature_count.items() if v >= 50}
    for index,key in enumerate(dep_feature_index.keys()):
        dep_feature_index[key] = index
    with open('./dep_feature/dep_feature_index.json', 'w', encoding='utf-8') as file:
        file.write(json.dumps(dep_feature_index, ensure_ascii=False))

In [23]:
x = []
y = []
x_sentence = []
with open('./dep_feature/dep_feature_index.json', 'r', encoding='utf-8') as file:
    dep_feature_index = json.load(file)
with open('./dep_feature/20250307_Yiyan_gpt_all_dep.json', 'r', encoding='utf-8') as file:
    instruct_dep_feature = json.load(file)
    for line in instruct_dep_feature:
        line_vec = np.zeros(len(dep_feature_index))
        for dep in line["deps_with_word"]:
            dep_list = dep.split()
            dep_feature = dep_list[0] + " " + dep_list[1] + " " + dep_list[2]
            if dep_feature in dep_feature_index:
                line_vec[dep_feature_index[dep_feature]] += 1
        x.append(line_vec)
        x_sentence.append(line["text"])
        y.append(1)
with open('./dep_feature/00LCMC_all_dep.json', 'r', encoding='utf-8') as file:
    origin_dep_feature = json.load(file)
    for line in origin_dep_feature:
        line_vec = np.zeros(len(dep_feature_index))
        for dep in line["deps_with_word"]:
            dep_list = dep.split()
            dep_feature = dep_list[0] + " " + dep_list[1] + " " + dep_list[2]
            if dep_feature in dep_feature_index:
                line_vec[dep_feature_index[dep_feature]] += 1
        x.append(line_vec)
        x_sentence.append(line["text"])
        y.append(0)
x = np.array(x)
y = np.array(y)

with open('./dep_feature/dep_feature_vec.json', 'w', encoding='utf-8') as file:
    file.write(json.dumps(x.tolist(), ensure_ascii=False))
with open('./dep_feature/dep_feature_label.json', 'w', encoding='utf-8') as file:
    file.write(json.dumps(y.tolist(), ensure_ascii=False))


In [ ]:
# from sklearn.metrics import f1_score

with open('./dep_feature/dep_feature_vec.json', 'r', encoding='utf-8') as file:
    x = json.load(file)
with open('./dep_feature/dep_feature_label.json', 'r', encoding='utf-8') as file:
    y = json.load(file)
x = np.array(x)
y = np.array(y)
# kf = KFold(n_splits=5, shuffle=True, random_state=42)
# 创建模型


skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
clf = RandomForestClassifier(n_estimators=100, random_state=42, max_features=300)
for fold, (train_index, test_index) in enumerate(skf.split(x,y)):
    X_train, X_test = x[train_index], x[test_index]
    y_train, y_test = y[train_index], y[test_index]

    clf.fit(X_train, y_train)
    score = clf.score(X_test, y_test)
    # y_pred = clf.predict(X_test)
    # f1 = f1_score(y_test, y_pred, average='micro')  # 可以选择 'micro', 'macro', 'weighted' 等
    # f2 = f1_score(y_test, y_pred, average='macro')  # 可以选择 'micro', 'macro', 'weighted' 等
    # f3 = f1_score(y_test, y_pred, average='weighted')  # 可以选择 'micro', 'macro', 'weighted' 等
    print(f"第 {fold+1} 折准确率: {score:.4f}")
    # print(f"第 {fold+1} 折f1-score: {f1:.4f}, f2-score: {f2:.4f}, f3-score: {f3:.4f}")

# 读取字典
with open('./dep_feature/dep_feature_index.json', 'r', encoding='utf-8') as f:
    feature_names = json.load(f)
# 反转字典：交换 key 和 value
feature_names = {v: k for k, v in feature_names.items()}

# 提取特征重要性
importances = clf.feature_importances_
indices = np.argsort(importances)[::-1]

# 打印特征重要性
print("特征重要性排名：")
for i in indices:
    print(f"{feature_names[i]}: {importances[i]:.4f}")

# 可视化
plt.figure(figsize=(8, 6))
plt.bar(range(10), importances[indices][:10], align='center', alpha=0.7)
plt.xticks(range(10), [feature_names[i] for i in indices[:10]], rotation=45)
plt.xlabel('特征名称')
plt.ylabel('重要性得分')
plt.title('特征重要性')
plt.tight_layout()
plt.show()

TypeError: array() missing required argument 'object' (pos 0)

In [ ]:
#PU, VV, punct作为决策点的每个节点，哪些句子被分到哪一类
# 获取所有树的决策路径，并记录哪些树使用了 PU, VV, punct 特征
decision_paths = []
dep_feature_index = json.load(open('./dep_feature/dep_feature_index.json', 'r', encoding='utf-8'))
feature_names = list(dep_feature_index.keys())

# 记录那些使用 PU, VV, punct 作为决策点的树和节点
for tree in clf.estimators_:
    for node_id in range(tree.tree_.node_count):
        feature_idx = tree.tree_.feature[node_id]  # 获取节点分裂的特征索引
        if feature_idx != -2:  # -2 表示叶子节点
            feature_name = feature_names[feature_idx]  # 获取特征的名称
            if feature_name in ['PN VV nsubj']:  # 仅关注 PU, VV, punct
                # 保存决策路径的详细信息
                decision_paths.append({
                    'tree': tree, 
                    'node_id': node_id, 
                    'feature': feature_name,
                    'threshold': tree.tree_.threshold[node_id]
                })

print(f"Found {len(decision_paths)} decision paths involving PU VV punct.")

Found 3308 decision paths involving PU, VV, punct.


In [ ]:
def find_decision_path(tree, node_id, feature_names):
    """
    回溯决策路径，返回从根节点到 node_id 的路径信息。
    
    :param tree: 训练好的 DecisionTreeClassifier
    :param node_id: 目标节点 ID
    :param feature_names: 特征名称列表（用于将特征索引转换为特征名称）
    :return: 决策路径（从根节点到指定节点）
    """
    path_trace = []
    
    while node_id != 0:  # 0 号节点是根节点
        parent_node = None
        for i in range(tree.tree_.node_count):
            # 判断是否是当前 node_id 的父节点
            if tree.tree_.children_left[i] == node_id or tree.tree_.children_right[i] == node_id:
                parent_node = i
                break

        if parent_node is None:
            path_trace.append(f"[WARNING] Could not find parent for Node {node_id}")
            break
        
        feature_idx = tree.tree_.feature[parent_node]
        threshold = tree.tree_.threshold[parent_node]

        # 确保特征索引有效
        feature_name = feature_names[feature_idx] if feature_idx != -2 else "Leaf Node"

        # 记录路径信息
        path_trace.append(f"Node {parent_node}: {feature_name} ≤ {threshold}")
        
        # 如果遇到 PU, VV, punct，停止回溯
        if feature_name in ['PU', 'VV', 'punct']:
            break
        
        # 继续回溯到上一个节点
        node_id = parent_node

    return list(reversed(path_trace))  # 反转路径，使其从根节点到目标节点

# 获取特征名称
feature_names = list(dep_feature_index.keys())  # 确保 dep_feature_index 是 {特征名: 索引} 的字典

# 遍历前 10 条记录，打印它们的决策路径
for result in results[:20]:
    sentence_index = result['sentence_index']
    predicted_class = result['predicted_class']
    feature = result['feature']
    threshold = result['threshold']
    tree = clf.estimators_[0]  # 选择第一棵决策树
    node_id = result['node_id']
    
    print(f"\nSentence {sentence_index} (Predicted class: {predicted_class})")
    print(f"  Passed through {feature} (Threshold: {threshold})")
    print(f"  Sentence: \"{result['sentence']}\"")
    print("  Decision Path:")

    # 获取该句子的决策路径
    decision_path = find_decision_path(tree, node_id, feature_names)

    # 打印路径（美化）
    if decision_path:
        for step in decision_path:
            print(f"    → {step}")
    else:
        print("    [INFO] No valid decision path found.")

    print("-" * 50)


Sentence 56781 (Predicted class: 0)
  Passed through PN VV nsubj (Threshold: 0.5)
  Sentence: "容积太小的冰箱，花了电费，冷藏食品却放不进多久甚至大一点的西瓜，或大盒奶油蛋糕放进去都会有困难，显然使用价值不大。"
  Decision Path:
    → Node 0: DEG PN assm ≤ 0.5
    → Node 1: VV VV comod ≤ 0.5
    → Node 2: NN NN nn ≤ 2.5
    → Node 3: NN NN root ≤ 0.5
--------------------------------------------------

Sentence 58503 (Predicted class: 0)
  Passed through PN VV nsubj (Threshold: 0.5)
  Sentence: "不然就是世界级笑话。"
  Decision Path:
    → Node 0: DEG PN assm ≤ 0.5
    → Node 1: VV VV comod ≤ 0.5
    → Node 2: NN NN nn ≤ 2.5
    → Node 3: NN NN root ≤ 0.5
--------------------------------------------------

Sentence 4984 (Predicted class: 1)
  Passed through PN VV nsubj (Threshold: 0.5)
  Sentence: "“你是说，讲故事？”"
  Decision Path:
    → Node 0: DEG PN assm ≤ 0.5
    → Node 1: VV VV comod ≤ 0.5
    → Node 2: NN NN nn ≤ 2.5
    → Node 3: NN NN root ≤ 0.5
--------------------------------------------------

Sentence 20781 (Predicted class: 1)
  Passed through 

In [24]:
# 用于记录每个句子通过 PU, VV, punct 特征分裂点的情况以及最终分类
results = []
# 随机选择 1000 行数据，并保留它们的索引
indices = np.random.choice(x.shape[0], size=1000, replace=False)

# 获取被选中的行数据
X_test = x[indices]

# 创建一个 (索引, 向量) 元组的列表
index_vector_pairs = list(zip(indices, X_test))



for i, sentence in index_vector_pairs:
    # 获取模型对句子的预测标签
    predicted_class = clf.predict([sentence])[0]
    # print("sentence: ", sentence)
    # print(f"Sentence {i} (Predicted class: {predicted_class})")

    # 遍历每棵树，检查该句子是否通过了包含 PU, VV, punct 的决策路径
    for path in decision_paths:
        tree = path['tree']
        node_id = path['node_id']
        feature_name = path['feature']
        threshold = path['threshold']
        # print("tree: ", tree)
        # print("node_id: ", node_id)
        # print("feature_name: ", feature_name)
        # print("threshold: ", threshold)

        # 获取句子对应特征的值
        feature_idx = dep_feature_index[feature_name]
        feature_value = sentence[feature_idx]

        # 判断该句子是否满足分裂条件
        if feature_value <= threshold:
            # 如果该句子通过了特定的决策点，记录该句子
            results.append({
                'sentence': x_sentence[i], 
                'sentence_index': i, 
                'feature': feature_name,
                'threshold': threshold,
                'predicted_class': predicted_class,
                'tree': tree,
                'node_id': node_id
            })
            break  # 找到该句子经过的一个决策路径后，跳出循环处理下一个句子

# 打印部分结果查看
for result in results[:10]:  # 这里只打印前10个句子的结果
    print(f"Sentence {result['sentence_index']} (Predicted class: {result['predicted_class']})")
    print(f"  Passed through {result['feature']} with threshold {result['threshold']}")
    print(f"  Sentence: {result['sentence']}")
    print("-" * 50)

Sentence 56781 (Predicted class: 0)
  Passed through PN VV nsubj with threshold 0.5
  Sentence: 容积太小的冰箱，花了电费，冷藏食品却放不进多久甚至大一点的西瓜，或大盒奶油蛋糕放进去都会有困难，显然使用价值不大。
--------------------------------------------------
Sentence 58503 (Predicted class: 0)
  Passed through PN VV nsubj with threshold 0.5
  Sentence: 不然就是世界级笑话。
--------------------------------------------------
Sentence 4984 (Predicted class: 1)
  Passed through PN VV nsubj with threshold 0.5
  Sentence: “你是说，讲故事？”
--------------------------------------------------
Sentence 20781 (Predicted class: 1)
  Passed through PN VV nsubj with threshold 0.5
  Sentence: 然后，一位神经科医生带着结果来找我。
--------------------------------------------------
Sentence 76288 (Predicted class: 0)
  Passed through PN VV nsubj with threshold 0.5
  Sentence: 穷虽然穷，杨干大还是有一点家底的，然而，这点积蓄是为了别的用场，积攒它，绝对不是为了有朝一日杨作新上学。
--------------------------------------------------
Sentence 48146 (Predicted class: 0)
  Passed through PN VV nsubj with threshold 1.5
  Sentence: 从中期来看，以色列欲使约旦河西岸和加

In [ ]:
# 寻找叶节点是PU VV punct的树
def find_decision_path_to_leaf(tree, node_id, feature_names):
    """
    查找从根节点到达叶节点的路径，并确保叶节点是 PU、VV 或 punct。

    :param tree: 训练好的 DecisionTreeClassifier
    :param node_id: 当前节点 ID
    :param feature_names: 特征名称列表
    :return: 从根节点到达叶节点的路径（包括特征名称和阈值）
    """
    path_trace = []

    # 回溯路径直到叶节点
    while tree.tree_.children_left[node_id] != tree.tree_.children_right[node_id]:  # 直到到达叶节点
        feature_idx = tree.tree_.feature[node_id]  # 获取当前节点使用的特征索引
        threshold = tree.tree_.threshold[node_id]  # 获取当前节点的阈值
        # print(feature_idx)
        # print(feature_names[feature_idx])
        
        # 获取特征名称
        feature_name = feature_names[feature_idx] if feature_idx != -2 else "Leaf Node"
        # print(feature_name)
        # 记录路径信息
        path_trace.append(f"Node {node_id}: {feature_name} <= {threshold}")
        
        # 根据条件判断进入左子树或右子树
        if tree.tree_.children_left[node_id] != -1:
            node_id = tree.tree_.children_left[node_id]  # 继续到左子树
        elif tree.tree_.children_right[node_id] != -1:
            node_id = tree.tree_.children_right[node_id]  # 继续到右子树
            

    # 获取叶节点的特征
    feature_idx = tree.tree_.feature[node_id]
    feature_name = feature_names[feature_idx] if feature_idx != -2 else "Leaf Node"
    print(feature_name)
    # 如果叶节点特征是 PU, VV 或 punct，记录该路径
    if feature_name in ['PN VV nsubj']:
        path_trace.append(f"Leaf Node {node_id}: {feature_name}")
        return path_trace
    return []  # 如果不是，返回空路径

# 获取特征名称（确保 dep_feature_index 是 {特征名: 索引} 的字典）
feature_names = list(dep_feature_index.keys())

# 遍历前 10 条记录，查找决策树叶节点为 PU、VV、punct 的句子
for result in results:
    sentence_index = result['sentence_index']
    predicted_class = result['predicted_class']
    feature = result['feature']
    threshold = result['threshold']
    tree = clf.estimators_[0]  # 选择第一棵决策树
    node_id = result['node_id']
    # print("sentence_index: ", sentence_index)
    # print(predicted_class)
    # print(feature)
    # print(tree)
    # print(node_id)
    # break
    
    
    # 获取该句子的决策路径
    decision_path = find_decision_path_to_leaf(tree, node_id, feature_names)

    # 如果找到符合条件的路径，才打印该句子
    if decision_path:
        print(f"\nSentence {sentence_index} (Predicted class: {predicted_class})")
        print(f"  Passed through {feature} (Threshold: {threshold})")
        print(f"  Sentence: \"{result['sentence']}\"")
        print("  Decision Path:")

        # 打印路径（美化）
        for step in decision_path:
            print(f"    → {step}")
        
        print("-" * 50)

Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node
Leaf Node


In [ ]:
# 暂时不用
# 获取所有树的决策路径，并记录哪些树使用了 PU, VV, punct 特征
decision_paths = []
feature_names = X.columns  # 假设 X 是一个 pandas DataFrame，包含了特征名称

# 记录那些使用 PU, VV, punct 作为决策点的树和节点
for tree in clf.estimators_:
    for node_id in range(tree.tree_.node_count):
        feature_idx = tree.tree_.feature[node_id]  # 分裂特征的索引
        if feature_idx != -2:  # -2 表示叶子节点
            feature_name = feature_names[feature_idx]  # 获取特征的名称
            if feature_name in ['PU', 'VV', 'punct']:  # 仅关注 PU, VV, punct
                decision_paths.append({
                    'tree': tree, 
                    'node_id': node_id, 
                    'feature': feature_name,
                    'threshold': tree.tree_.threshold[node_id]
                })

print(f"Found {len(decision_paths)} decision paths involving PU, VV, punct.")

In [ ]:
# 暂时不用
# # 遍历测试数据集中的每个句子，判断其是否通过了包含 PU, VV, punct 的决策点
for i, sentence in enumerate(X_test.values):
    prediction = clf.predict([sentence])  # 获取模型对该句子的预测

    # 遍历每棵树，检查该句子是否经过了包含 PU, VV, punct 的决策路径
    for path in decision_paths:
        tree = path['tree']
        node_id = path['node_id']
        feature_name = path['feature']
        threshold = path['threshold']

        # 检查该句子是否符合分裂条件（特征值与阈值进行比较）
        feature_idx = X_test.columns.get_loc(feature_name)  # 获取特征的列索引
        feature_value = sentence[feature_idx]
        
        if feature_value <= threshold:
            print(f"Sentence {i} went through a decision path with {feature_name} <= {threshold}.")
            print(f"Sentence: {sentence}")
            break  # 如果找到了决策路径，打印句子并跳到下一个句子